In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip -q install transformers datasets scikit-learn torch pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
import scipy.linalg

/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

In [5]:
def get_bert_embeddings(model, data_loader, device):
    model = model.eval()
    embeddings = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state
            cls_embeddings = hidden_states[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

def get_rowspace_projection(W):
    if W.ndim == 1:
        W = W.reshape(1, -1)
    
    basis = scipy.linalg.orth(W.T)
    P_row = basis @ basis.T
    return P_row

def inlp(X, Z, n_iterations):
    X_projected = X.copy()
    P_final = np.eye(X.shape[1])
    
    for i in range(n_iterations):
        clf = LinearSVC(dual='auto', max_iter=2000)
        clf.fit(X_projected, Z)
        W = clf.coef_
        
        P_row = get_rowspace_projection(W)
        P_null = np.eye(X.shape[1]) - P_row
        
        P_final = P_null @ P_final
        X_projected = X_projected @ P_null.T
        
    return P_final, X_projected

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

In [7]:
df_sample = pd.read_csv('Jigsaw data processing/inlp_subset.csv')

texts = df_sample['comment_text'].values
z = df_sample['Z'].values

# Load precomputed embeddings
X = np.load('Jigsaw data processing/bert_embeddings.npz')['X']

In [8]:
X_train, X_test, z_train, z_test = train_test_split(
    X, z, test_size=0.3, random_state=42
)

In [9]:
gender_clf = LogisticRegression(max_iter=1000)
gender_clf.fit(X_train, z_train)
z_pred_orig = gender_clf.predict(X_test)
print(f"Original Accuracy (Subset Z): {accuracy_score(z_test, z_pred_orig):.4f}")

/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

Original Accuracy (Subset Z): 0.8254


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [ ]:
P, X_train_inlp = inlp(X_train, z_train, n_iterations=50)
X_test_inlp = X_test @ P.T

/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: divide by zero encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: overflow encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:34: RuntimeWarning: invalid value encountered in matmul
  P_final = P_null @ P_final
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: divide by zero encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: overflow encountered in matmul
  X_projected = X_projected @ P_null.T
/var/folders/2l/chysr0_53xx2gf90f24f72jc0000gn/T/ipykernel_22535/1152264484.py:35: RuntimeWarning: invalid value encountered in matmul
  X_projected = X_projected @ P_n

In [ ]:
gender_clf_inlp = LogisticRegression(max_iter=1000)
gender_clf_inlp.fit(X_train_inlp, z_train)
z_pred_inlp = gender_clf_inlp.predict(X_test_inlp)
print(f"INLP Accuracy (Subset Z): {accuracy_score(z_test, z_pred_inlp):.4f}")

/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packag

INLP Accuracy (Subset Z): 0.6416


/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/elieattali/Desktop/ENSAE/2A/StatApp/Stat_App/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
